# Week 10 skills notebook: samples and uncertainty intervals

This notebook belongs to the skills track. It uses a made-up courier company so that the skills stay separate from this week's project.

The company handled 20,000 parcels last month. In this notebook you can see every one of them, which never happens in real work. That lets you check each guess a sample makes against the true answer.

You will make an estimate from a sample, watch it change from sample to sample, measure that change, catch an AI that mixes up two kinds of spread, build a bootstrap interval, and test what "95 percent" promises.

**How to use it:** run each cell in order with the play button or Shift+Enter. Read the output before you move on. Two "your turn" tasks near the end have CHECK cells.

## 1. The population you normally never see

The **population** is everything you want to know about: here, all 20,000 parcels. The number you want is the **target**, for example the mean weight of all parcels or the share that arrived late.

In real work nobody weighs every parcel. Here we can, so the next cell prints the true answers. Keep them in mind: every later guess is judged against them.

The company's log lists the parcels depot by depot: first the city depot, then the airport, then the harbour. That order matters in section 5.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(10)
N = 20000
depot = rng.choice(["city", "airport", "harbour"], size=N, p=[0.4, 0.35, 0.25])
median_kg = pd.Series(depot).map({"city": 1.5, "airport": 3.0, "harbour": 5.0}).to_numpy()
weight = np.round(median_kg * rng.lognormal(0, 0.6, size=N), 2)
late_chance = pd.Series(depot).map({"city": 0.05, "airport": 0.12, "harbour": 0.25}).to_numpy()
late = rng.random(N) < late_chance

# the log lists the city depot first, then the airport, then the harbour
order = pd.Series(depot).map({"city": 0, "airport": 1, "harbour": 2}).to_numpy()
parcels = pd.DataFrame({"depot": depot, "weight_kg": weight, "late": late})
parcels = parcels.iloc[np.argsort(order, kind="stable")].reset_index(drop=True)

true_mean = parcels["weight_kg"].mean()
true_late = parcels["late"].mean()
print("Parcels in the population:", len(parcels))
print("Population mean weight (kg):", round(true_mean, 2))
print("Population late share:", round(true_late, 3))
parcels.head()

## 2. One sample, one guess

A manager cannot weigh 20,000 parcels, so she weighs a random **sample** of 100. The rule "take the mean of the sample" is an **estimator**. The number it gives on this sample is an **estimate**.

`random_state=6` is a seed for this one draw, so your sample is the same as everyone else's.

In [ ]:
sample = parcels.sample(n=100, random_state=6)
estimate = sample["weight_kg"].mean()
print("Sample mean weight (kg):", round(estimate, 2))
print("Sample late share:", round(sample["late"].mean(), 3))
print("Miss on the mean weight (kg):", round(estimate - true_mean, 2))

## 3. Another sample, another guess

Draw five more samples of 100 with other seeds. Each one gives a different estimate, and none of them is wrong in the sense of a mistake. The change from sample to sample is the price of not weighing everything.

In [ ]:
for seed in range(1, 6):
    other = parcels.sample(n=100, random_state=seed)
    print("sample", seed, "mean weight (kg):", round(other["weight_kg"].mean(), 2))

## 4. The estimator's test bench

Because we know the population, we can test the estimator itself. The next cell draws 2,000 samples of 100 parcels and keeps each sample's mean. Two questions matter.

- **Where is the centre?** If the 2,000 means centre on the true mean, the estimator has no **bias**.
- **How wide is the spread?** The typical distance of a sample mean from the centre is the **standard error**. It tells you how far one estimate is likely to sit from the truth.

In [ ]:
import matplotlib.pyplot as plt

w = parcels["weight_kg"].to_numpy()
means = np.array([rng.choice(w, size=100, replace=False).mean() for _ in range(2000)])
print("Average of 2,000 sample means (kg):", round(means.mean(), 2))
print("Spread of the 2,000 sample means, the standard error (kg):", round(means.std(), 3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(means, bins=40)
ax.axvline(true_mean, color="black", linestyle="--", label="population mean")
ax.set_title("2,000 sample means, each from 100 parcels")
ax.set_xlabel("sample mean weight (kg)")
ax.set_ylabel("number of samples")
ax.legend()
plt.tight_layout()
plt.show()

## 5. A second estimator: the first rows of the log

A clerk saves time by weighing the first parcels in the log instead of a random sample. That is also an estimator, and we can test it the same way.

The next cell compares the clerk's method with a random sample of the same size, for 100, 400 and 1,600 parcels. Watch what happens to each one as the sample grows.

In [ ]:
print("Population mean weight (kg):", round(true_mean, 2))
for n in [100, 400, 1600]:
    first_rows = w[:n].mean()
    random_rows = rng.choice(w, size=n, replace=False).mean()
    print(f"{n} parcels: first rows of the log {first_rows:.2f} kg, random sample {random_rows:.2f} kg")

The first rows all come from the city depot, where parcels are lighter. More rows from the same place do not move the clerk's answer towards the truth. **A larger sample shrinks the standard error, and it leaves bias where it was.** No interval in the rest of this notebook can repair a sample that was not drawn at random.

## 6. Two kinds of spread

Two numbers measure spread, and they answer different questions.

- **The standard deviation** describes the world: how much single parcels differ from each other. A bigger sample measures it more precisely, and it does not shrink.
- **The standard error** describes your doubt about the estimate: how much the sample mean would move with a new sample. It equals the standard deviation divided by the square root of the sample size, so it shrinks as the sample grows.

The cell below draws 2,000 samples at each of three sizes: 25, 100 and 400 parcels. For each size it prints the typical standard deviation inside a sample and the standard error of the sample mean, and it checks the formula against the simulation.

In [ ]:
print("Standard deviation of all 20,000 weights (kg):", round(w.std(), 2))
for n in [25, 100, 400]:
    draws = [rng.choice(w, size=n, replace=False) for _ in range(2000)]
    typical_sd = np.mean([d.std(ddof=1) for d in draws])
    simulated_se = np.std([d.mean() for d in draws])
    print(f"n = {n}: typical standard deviation {typical_sd:.2f} kg, standard error {simulated_se:.3f} kg by simulation and {w.std() / np.sqrt(n):.3f} kg by formula")

Four times as many parcels halve the standard error, because the square root of 4 is 2. That is the **square-root law**, and it is why a sample that is twice as precise costs four times as much work.

## 7. Read code an AI assistant wrote

We gave an AI assistant the sample of 100 weights and asked: *"Give me a 95% interval for the average parcel weight."* It wrote the cell below. Run it, then decide whether the answer is right.

In [ ]:
# Written by an AI assistant
m = sample["weight_kg"].mean()
sd = sample["weight_kg"].std()
print("95% interval for the average weight:", round(m - 1.96 * sd, 2), "to", round(m + 1.96 * sd, 2), "kg")

The code runs, and the interval starts below zero kilograms, which no parcel can weigh. The assistant used the standard deviation where the standard error belongs. Its range roughly describes where single parcels fall. It says almost nothing about where the mean is.

The next cell builds the interval with the standard error and measures both against the population.

In [ ]:
se = sd / np.sqrt(len(sample))
print("Standard error (kg):", round(se, 3))
print("Interval with the standard error:", round(m - 1.96 * se, 2), "to", round(m + 1.96 * se, 2), "kg")
print("The AI's interval is wider by a factor of:", round(sd / se, 1))
inside = ((w >= m - 1.96 * sd) & (w <= m + 1.96 * sd)).mean()
print("Share of single parcels inside the AI's interval:", round(inside, 3))

A 95% interval for the mean should hold the true mean in about 95 of every 100 samples. The next cell tests both methods on 1,000 fresh samples and counts how often each interval holds the true mean.

In [ ]:
ai_hits, se_hits = 0, 0
for _ in range(1000):
    s = rng.choice(w, size=100, replace=False)
    mm, ss = s.mean(), s.std(ddof=1)
    ai_hits += int(mm - 1.96 * ss <= true_mean <= mm + 1.96 * ss)
    se_hits += int(mm - 1.96 * ss / np.sqrt(100) <= true_mean <= mm + 1.96 * ss / np.sqrt(100))
print("Of 1,000 samples, the AI's interval held the true mean:", ai_hits)
print("Of 1,000 samples, the standard-error interval held the true mean:", se_hits)

# CHECK: the AI's interval almost never misses, because it is far too wide; the other is near 95 percent
assert ai_hits > 990 and 900 < se_hits < 990
print("CHECK passed")

An interval that never misses sounds safe. It is useless, because it is ten times wider than the evidence allows. An honest interval is as narrow as the data supports and misses about as often as it says it will.

## 8. The bootstrap

The formula in section 7 assumes the sample mean behaves in a regular way. When you are unsure, or when no formula fits your estimate, the **bootstrap** gives an interval by computation.

1. Draw a new sample of 100 from your own sample, **with replacement**, so some parcels appear twice and others not at all.
2. Compute the estimate on that resample.
3. Repeat 2,000 times, and keep the middle 95% of the results.

Drawing with replacement is the point. Drawn without replacement, every resample would be your sample in a new order, and all 2,000 estimates would be identical.

In [ ]:
x = sample["weight_kg"].to_numpy()
boot_means = np.array([rng.choice(x, size=len(x), replace=True).mean() for _ in range(2000)])
boot_lo, boot_hi = np.percentile(boot_means, [2.5, 97.5])
print("Bootstrap 95% interval for the mean weight:", round(boot_lo, 2), "to", round(boot_hi, 2), "kg")
print("Bootstrap standard error (kg):", round(boot_means.std(), 3))

## 9. What 95 percent promises

The next cell repeats the whole procedure 200 times: a fresh sample of 100, then its own bootstrap interval. It counts how many of the 200 intervals hold the true mean, and draws them all. Red lines are the misses.

In [ ]:
intervals = []
for _ in range(200):
    s = rng.choice(w, size=100, replace=False)
    resamples = s[rng.integers(0, 100, size=(1000, 100))].mean(axis=1)
    intervals.append(np.percentile(resamples, [2.5, 97.5]))
intervals = np.array(intervals)
covers = (intervals[:, 0] <= true_mean) & (true_mean <= intervals[:, 1])
print("Bootstrap intervals that held the true mean:", int(covers.sum()), "of 200")

fig, ax = plt.subplots(figsize=(8, 5))
for i, (lo, hi) in enumerate(intervals):
    ax.plot([lo, hi], [i, i], color="tab:blue" if covers[i] else "tab:red", linewidth=1)
ax.axvline(true_mean, color="black", linestyle="--")
ax.set_title("200 bootstrap intervals; the red ones miss the population mean")
ax.set_xlabel("mean weight (kg)")
ax.set_ylabel("sample number")
plt.tight_layout()
plt.show()

## 10. Reading an interval honestly

Once you have drawn your sample and computed your interval, the true mean is either inside it or not. Nothing about that one interval is random any more. **The 95 percent describes the method:** used again and again on fresh samples, it holds the truth about 95 times in 100, as the chart above shows.

A common misreading says "there is a 95 percent chance that the true mean lies in this interval". A safer sentence names the sample, the estimate and the method: *"We weighed a random sample of 100 parcels. Their mean weight was X kg, and a method that holds the true mean in about 95 of 100 samples gives an interval from A to B kg."*

An interval also covers only one kind of error: the luck of the random draw. It cannot see bias. The clerk's sample from section 5 would get a narrow interval around the wrong answer.

## 11. Your turn: an interval for the late share

A share is the mean of a column of 0s and 1s, so the bootstrap works on it the same way.

Ask your AI assistant for code that draws 2,000 bootstrap resamples of `sample["late"]` and stores the 2.5th and 97.5th percentiles of the resampled shares in `late_lo` and `late_hi`. Paste its code into the cell below and run the CHECK cell.

Before you trust the code, check two things: that it resamples **with** replacement, and that every resample has 100 rows.

In [ ]:
# your turn 1: store the bootstrap interval for the late share in late_lo and late_hi
late_lo, late_hi = None, None

In [ ]:
# CHECK 1
p_hat = sample["late"].mean()
formula_width = 2 * 1.96 * np.sqrt(p_hat * (1 - p_hat) / len(sample))
assert late_lo is not None and late_hi is not None, "late_lo or late_hi is still empty"
assert late_lo < p_hat < late_hi, "the interval must contain the sample's own share"
assert 0.6 * formula_width < late_hi - late_lo < 1.6 * formula_width, "the width is far from what the standard error predicts"
print("CHECK 1 passed: late share", round(p_hat, 2), "with interval", round(late_lo, 3), "to", round(late_hi, 3))

## 12. Your turn: how many parcels for a tighter answer?

The manager wants the late share to within 2 percentage points either way, so the half-width of the interval must be at most 0.02.

The half-width of a 95% interval for a share is 1.96 times the standard error, and the standard error of a share p from n parcels is the square root of p(1 - p)/n. Use your sample's late share as p, solve for n, and store the smallest whole number that is large enough in `n_needed`.

The CHECK cell tests your answer the honest way: it draws 1,000 samples of that size from the population and counts how often the sample share lands within 0.02 of the truth.

In [ ]:
# your turn 2: store the smallest sample size with a half-width of at most 0.02 in n_needed
n_needed = None

In [ ]:
# CHECK 2
assert n_needed is not None, "n_needed is still empty"
p_hat = sample["late"].mean()
assert abs(n_needed - 1.96 ** 2 * p_hat * (1 - p_hat) / 0.02 ** 2) <= 1, "use the formula with the sample's late share"
lv = parcels["late"].to_numpy().astype(float)
shares = np.array([rng.choice(lv, size=int(n_needed), replace=False).mean() for _ in range(1000)])
within = np.mean(np.abs(shares - true_late) <= 0.02)
assert within > 0.9
print("CHECK 2 passed: n_needed =", int(n_needed), "and", round(within, 3), "of the samples land within 0.02 of the true late share")

## 13. Your turn: three sentences to repair

Each sentence below misreads an interval. In a text cell under this one, say what is wrong with each, then rewrite it as a correct claim using the numbers from section 8.

1. "There is a 95 percent chance that the true mean weight lies inside our interval."
2. "95 percent of parcels weigh between the two ends of our interval."
3. "Our interval is narrow, so our sample cannot be biased."

Then ask your AI assistant to judge the same three sentences, and note where you and it disagree.